# 5-1. 재시도 정책

선택 심화. `5`를 마친 뒤 본다.

`5`는 예산과 오류 문구처럼 실험에서 바뀌는 값을 `CONFIG`로 뺐다. 재시도 정책도 같은 자리에 둔다. 여기서는 그 설정에 어떤 값을 넣어야 하는지 잰다.
일정 확률로 실패하는 도구 위에서 네 가지 정책을 돌려 성공률과 비용을 비교하고, 부작용이 있는 도구를 재시도할 때 생기는 중복 실행을 막는다.

외부요소로 인한 에러 예시
ex> 429 에러 : 호출을 너무 많이 해서, 충전된 API LIMIT에 도달


### 공통 설정

`5`와 같은 블록이다. 실패는 시드를 고정한 난수로 만든다.
429 의 쿨다운만은 실제 시간으로 재므로, 모델 응답이 오가는 시간에 따라 실패 순서가 실행마다 조금 달라질 수 있다.

In [1]:
import hashlib
import json
import os
import random
import re
import shutil
import time
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]

client = OpenAI()

PRICES = {MODEL: (2.50, 15.00)}
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

### 실패는 세 종류다

외부 도구가 실패하는 방식은 대개 셋이다. 응답이 늦어 끊기는 타임아웃, 요청 한도를 넘긴 429, 서버 쪽 일시 오류(503).
셋 다 같은 인자로 다시 부르면 성공할 수 있다. 다만 429 는 서버가 알려 준 시간만큼 기다리기 전에는 몇 번을 불러도 다시 429 다.

In [2]:
class ToolTimeout(Exception):
    pass


class Transient(Exception):
    pass


class RateLimited(Exception):
    def __init__(self, wait):
        super().__init__(f"429 요청 한도 초과. {wait}초 뒤 다시 시도")
        self.wait = wait


RETRYABLE = (ToolTimeout, Transient, RateLimited)

`Flaky`는 도구 함수를 감싸 정해진 확률로 세 실패 중 하나를 낸다. 429 를 한 번 내면 `cooldown` 초 동안은 무엇을 불러도 429 다.
실험이 짧게 끝나도록 실패 확률을 실제보다 높게 잡았다.

In [3]:
class Flaky:
    def __init__(self, seed, p=0.4, cooldown=1.0):
        self.rng, self.p, self.cooldown = random.Random(seed), p, cooldown
        self.until, self.calls, self.log = 0.0, 0, []

    def wrap(self, fn):
        def tool(**args):
            self.calls += 1
            now = time.monotonic()
            if now < self.until:
                self.log.append("429")
                raise RateLimited(round(self.until - now, 1))
            x = self.rng.random()
            if x < self.p / 3:
                self.log.append("타임아웃")
                raise ToolTimeout("도구 응답 시간 초과")
            if x < self.p * 2 / 3:
                self.until = now + self.cooldown
                self.log.append("429")
                raise RateLimited(self.cooldown)
            if x < self.p:
                self.log.append("503")
                raise Transient("503 일시 오류")
            self.log.append("성공")
            return fn(**args)
        return tool

### 도구

`5`의 `grep_docs` 대신 낱말이 포함된 문서 수를 바로 돌려주는 `count_docs` 하나를 쓴다. 재시도가 아닌 다른 이유로 답이 틀리는 일을 줄이기 위해서다.
빈 낱말은 인자 오류로 거부한다. 이것은 다시 불러도 성공하지 않는 실패다.

In [4]:
CORPUS = Path("../../../../../data/corpus/graph")


def count_docs(pattern):
    if not pattern.strip():
        raise ValueError("빈 낱말은 셀 수 없다")
    return sum(pattern in f.read_text(encoding="utf-8") for f in sorted(CORPUS.glob("*.md")))


def spec(name, desc, props):
    return {"type": "function", "name": name, "description": desc, "strict": True,
            "parameters": {"type": "object", "properties": props,
                           "required": list(props), "additionalProperties": False}}


TOOLS = [spec("count_docs", "낱말이 들어 있는 문서 수를 돌려준다.",
              {"pattern": {"type": "string", "description": "찾을 낱말"}})]
IMPL = {"count_docs": count_docs}

시드 0 으로 열 번 불러 실패가 어떻게 섞이는지 본다. 429 뒤에 곧바로 부른 것은 다시 429 다.

In [5]:
flaky = Flaky(seed=0)   # 결과가 동일하게 나오길 기대
tool = flaky.wrap(count_docs)
for _ in range(10):
    try:
        tool(pattern="광복군")
    except RETRYABLE:
        pass
print(flaky.log)

['성공', '성공', '성공', '429', '429', '429', '429', '429', '429', '429']


### 코어 블록

`5`의 코어를 그대로 복제한다. 바뀐 것은 `CONFIG`에 `retry` 항목 하나가 늘어난 것뿐이다.

In [6]:
CONFIG = {
    "instructions": "도구로 확인한 것만 답한다. 세지 않은 숫자는 쓰지 않는다.",
    "max_turns": 6, "max_tokens": 60000, "max_seconds": 90,
    "gated": ["write_note"],
    "gate_error": "error: 이 도구는 승인이 필요하다. 무엇을 할지 사용자에게 먼저 확인한다.",
    "tool_error": "error: {e}. 인자를 고쳐 다시 부른다.",
    "retry": {"max": 0, "base": 0.0, "honor_after": False},     # RETRY 추가
}

In [ ]:
# LLM이 재시도에 관한 걸 알지 않아도 괜찮아 변경사항 없음
def call_tool(name, args, approve):             # ToolRegistry
    if name in CONFIG["gated"] and not approve:  # Policy
        return CONFIG["gate_error"]
    try:
        return json.dumps(IMPL[name](**args), ensure_ascii=False)
    except Exception as e:
        return CONFIG["tool_error"].format(e=e)


def run(task, approve=False):                   # Runtime
    messages = [{"role": "user", "content": task}]          # Session
    trace, t0, used = [], time.perf_counter(), 0
    for turn in range(1, CONFIG["max_turns"] + 1):
        r = keep(client.responses.create(model=MODEL, instructions=CONFIG["instructions"],
                                         input=messages, tools=TOOLS, store=False))
        used += r.usage.total_tokens
        calls = [i for i in r.output if i.type == "function_call"]
        trace.append({"턴": turn, "도구": [c.name for c in calls], "토큰": used,
                      "초": round(time.perf_counter() - t0, 1)})            # Trace
        if not calls:
            return r.output_text, "완료", trace
        if used > CONFIG["max_tokens"]:
            return None, "토큰 예산", trace
        if time.perf_counter() - t0 > CONFIG["max_seconds"]:
            return None, "시간 예산", trace
        messages += [i.model_dump() for i in r.output]
        messages += [{"type": "function_call_output", "call_id": c.call_id,
                      "output": call_tool(c.name, json.loads(c.arguments), approve)}
                     for c in calls]
    return None, "반복 예산", trace

### 재시도는 코어 밖에서 감싼다

재시도를 `call_tool` 안에 넣으면 코어가 바뀌고 복제본을 전부 고쳐야 한다.
그 대신 `IMPL`에 넣는 함수를 감싼다. 코어는 감싼 함수를 그냥 부른다.

`with_retry`는 `CONFIG["retry"]`를 부를 때마다 읽는다. `max`는 다시 부르는 횟수, `base`는 첫 대기 시간이고 한 번 실패할 때마다 두 배로 는다.
실패한 뒤 바로 다시 부르지 않고 기다렸다 부르는 것을 백오프라고 하고, 이렇게 대기 시간을 두 배씩 늘리면 지수 백오프다.
`honor_after`가 켜져 있으면 429 가 알려 준 대기 시간보다 짧게 기다리지 않는다.
`RETRYABLE`에 없는 예외는 다시 부르지 않고 그대로 올려 보낸다.

In [8]:
def with_retry(fn):
    def tool(**args):
        r = CONFIG["retry"]     # retry 횟수
        for attempt in range(r["max"] + 1):
            try:
                return fn(**args)
            except RETRYABLE as e:
                if attempt == r["max"]:
                    raise
                wait = r["base"] * 2 ** attempt
                if r["honor_after"] and isinstance(e, RateLimited):
                    wait = max(wait, e.wait)
                time.sleep(wait)
    return tool

인자 오류는 다시 부르지 않는다. 재시도 횟수를 넉넉히 줘도 실행은 한 번이다.

In [9]:
CONFIG["retry"] = {"max": 3, "base": 0.1, "honor_after": True}
calls = []
tool = with_retry(lambda **a: calls.append(1) or count_docs(**a))
try:
    tool(pattern=" ")
except ValueError as e:
    print("실행", len(calls), "번 |", e)

실행 1 번 | 빈 낱말은 셀 수 없다


### 과제와 채점

낱말 여덟 개의 문서 수를 센다. 도구 호출이 여덟 번이면 실패도 몇 번은 섞인다. 정답은 같은 함수로 만든다.

In [10]:
TERMS = ["한국독립당", "광복군", "독립신문", "하와이", "의열단", "신민회", "조선민족혁명당", "신한청년당"]
TRUTH = {w: count_docs(w) for w in TERMS}
TASK = "아래 낱말 각각이 나오는 문서 수를 세어 '낱말: 숫자' 꼴로 한 줄씩 답한다.\n" + "\n".join(TERMS)
print(TRUTH)

{'한국독립당': 16, '광복군': 13, '독립신문': 11, '하와이': 11, '의열단': 10, '신민회': 9, '조선민족혁명당': 8, '신한청년당': 6}


낱말이 든 줄에서 낱말 뒤의 첫 숫자를 읽는다. 맞은 수와 틀린 수를 따로 센다. 숫자를 쓰지 않은 것은 어느 쪽에도 넣지 않는다.

In [11]:
def grade(text):
    ok = wrong = 0
    for w, n in TRUTH.items():
        line = next((l for l in (text or "").splitlines() if w in l), "")
        m = re.search(r"\d+", line.split(w, 1)[-1]) if line else None
        if m and int(m[0]) == n:
            ok += 1
        elif m:
            wrong += 1
    return ok, wrong

### 네 가지 정책

모두 `CONFIG`만 다르다.

- 재시도 없음: `5`의 기본값이다. 오류 문구는 "인자를 고쳐 다시 부른다"다.
- 바로 재시도: 기다리지 않고 세 번까지 다시 부른다.
- 지수 백오프: 0.25초에서 시작해 두 배씩 기다리고, 429 의 대기 시간을 지킨다.
- 모델에 맡김: 하네스는 다시 부르지 않고, 오류 문구로 모델에게 같은 인자로 다시 부르라고 한다.

In [12]:
BASE = json.loads(json.dumps(CONFIG))
POLICIES = {
    "재시도 없음": {"retry": {"max": 0, "base": 0.0, "honor_after": False}},
    "바로 재시도": {"retry": {"max": 3, "base": 0.0, "honor_after": False}},    # 3번까지 재시도
    "지수 백오프": {"retry": {"max": 3, "base": 0.25, "honor_after": True}},    # 일정시간 기다린 후 3회까지 재시도 => 또 발생 시 기다리는 시간을 배수시킴
    "모델에 맡김": {"retry": {"max": 0, "base": 0.0, "honor_after": False},
                "tool_error": "error: {e}. 일시적인 실패다. 같은 인자로 다시 부른다."}, # 모델에게 시킴
}


def use(policy):
    CONFIG.clear()
    CONFIG.update(json.loads(json.dumps(BASE)))
    CONFIG.update(POLICIES[policy])

한 번 돌리는 함수다. 정책끼리 같은 시드로 비교해 실패를 되도록 같은 조건으로 맞춘다.

In [13]:
def attempt(policy, seed, task=TASK):
    use(policy)
    flaky = Flaky(seed)
    IMPL["count_docs"] = with_retry(flaky.wrap(count_docs))
    mark, t0 = len(USAGE), time.perf_counter()
    answer, why, trace = run(task)
    ok, wrong = grade(answer)
    return {"정답": ok, "오답": wrong, "사유": why, "턴": len(trace), "도구 실행": flaky.calls,
            "호출": len(USAGE) - mark, "토큰": sum(u.input_tokens + u.output_tokens for _, u in USAGE[mark:]),
            "초": time.perf_counter() - t0, "실패": flaky.log, "답": answer}

시드 1 은 첫 턴에 429 가 일찍 나와 쿨다운 동안 나머지 호출이 모두 막히는 경우다. 기본값으로 돌려 본다.

In [14]:
r = attempt("재시도 없음", seed=1)
print(r["실패"])
print(r["사유"], "|", (r["답"] or "").replace("\n", " / ")[:120])

['429', '429', '429', '429', '429', '429', '429', '429', '성공', '성공', '429', '성공', '성공']
반복 예산 | 


정책마다 시드 다섯 개로 돌린다.

In [15]:
SEEDS = range(5)
RESULTS = {}
for policy in POLICIES:
    RESULTS[policy] = [attempt(policy, s) for s in SEEDS]
    for s, r in zip(SEEDS, RESULTS[policy]):
        print(f"{policy}  시드 {s}  {r['정답']}/8  오답 {r['오답']}  {r['사유']}  턴 {r['턴']}"
              f"  도구 실행 {r['도구 실행']:2d}  {r['초']:4.1f}초")

재시도 없음  시드 0  0/8  오답 0  반복 예산  턴 6  도구 실행 13   8.0초
재시도 없음  시드 1  0/8  오답 0  반복 예산  턴 6  도구 실행 13   8.4초
재시도 없음  시드 2  8/8  오답 0  완료  턴 3  도구 실행 11   5.3초
재시도 없음  시드 3  0/8  오답 0  반복 예산  턴 6  도구 실행 13   8.1초
재시도 없음  시드 4  0/8  오답 0  완료  턴 4  도구 실행 10   6.7초
바로 재시도  시드 0  0/8  오답 0  반복 예산  턴 6  도구 실행 29   8.8초
바로 재시도  시드 1  0/8  오답 0  반복 예산  턴 6  도구 실행 40   8.5초
바로 재시도  시드 2  8/8  오답 0  완료  턴 2  도구 실행 11   3.6초
바로 재시도  시드 3  0/8  오답 0  반복 예산  턴 6  도구 실행 43   8.5초
바로 재시도  시드 4  0/8  오답 0  완료  턴 3  도구 실행 36   4.8초
지수 백오프  시드 0  8/8  오답 0  완료  턴 2  도구 실행 10   4.9초
지수 백오프  시드 1  8/8  오답 0  완료  턴 2  도구 실행 12   7.3초
지수 백오프  시드 2  8/8  오답 0  완료  턴 2  도구 실행 11   5.4초
지수 백오프  시드 3  8/8  오답 0  완료  턴 2  도구 실행 14   7.9초
지수 백오프  시드 4  7/8  오답 1  완료  턴 4  도구 실행 20  14.4초
모델에 맡김  시드 0  0/8  오답 0  반복 예산  턴 6  도구 실행 13   8.9초
모델에 맡김  시드 1  0/8  오답 0  반복 예산  턴 6  도구 실행 13   9.4초
모델에 맡김  시드 2  7/8  오답 1  완료  턴 3  도구 실행 11   5.2초
모델에 맡김  시드 3  0/8  오답 0  반복 예산  턴 6  도구 실행 13   8.4초
모델에 맡김  시드 4  0/8  오답 0

### 나란히 둔다

"전부 맞음"은 여덟 낱말을 다 맞힌 실행 수다. 비용은 모델 호출만 센다. 도구 실행은 공짜라고 본다.

In [16]:
def dollars(rows):
    total = 0.0
    for name, u in rows:
        p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
        if p:
            total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]
    return total


n = len(SEEDS)
for policy, rows in RESULTS.items():
    print(f"{policy}  전부 맞음 {sum(r['정답'] == 8 for r in rows)}/{n}"
          f"  평균 정답 {sum(r['정답'] for r in rows) / n:3.1f}  오답 {sum(r['오답'] for r in rows)}"
          f"  턴 {sum(r['턴'] for r in rows) / n:3.1f}  모델 호출 {sum(r['호출'] for r in rows):2d}"
          f"  토큰 {sum(r['토큰'] for r in rows):6,d}  도구 실행 {sum(r['도구 실행'] for r in rows):3d}"
          f"  {sum(r['초'] for r in rows) / n:4.1f}초/회")

재시도 없음  전부 맞음 1/5  평균 정답 1.6  오답 0  턴 5.0  모델 호출 25  토큰 14,878  도구 실행  60   7.3초/회
바로 재시도  전부 맞음 1/5  평균 정답 1.6  오답 0  턴 4.6  모델 호출 23  토큰 13,387  도구 실행 159   6.8초/회
지수 백오프  전부 맞음 4/5  평균 정답 7.8  오답 1  턴 2.4  모델 호출 12  토큰  4,871  도구 실행  67   8.0초/회
모델에 맡김  전부 맞음 0/5  평균 정답 1.4  오답 1  턴 5.2  모델 호출 26  토큰 16,403  도구 실행  61   7.8초/회


바로 재시도는 타임아웃과 503 은 넘기지만 429 는 넘기지 못한다. 쿨다운 안에서 세 번을 다 써 버리고, 도구 실행 수만 늘린다.
지수 백오프는 429 의 대기 시간을 지키므로 같은 시드에서도 실패가 모델까지 올라가지 않는다. 기다린 시간만큼 한 번 실행이 길어진다.

모델에 맡기면 모델이 같은 인자로 다시 부른다. 그러나 모델은 기다릴 수 없다. 다음 턴은 곧바로 오고, 쿨다운 안이면 다시 429 다.
그 사이 턴과 토큰이 늘고, 시드에 따라서는 반복 예산을 다 쓰거나 확인하지 못했다고 답하고 끝낸다. 대기가 필요한 실패는 하네스가 맡아야 한다.

재시도 없음과 모델에 맡김의 결과가 거의 같은 것도 볼 만하다. "인자를 고쳐 다시 부른다"는 문구를 받아도 모델은 인자를 고치지 않고 같은 인자로 다시 불렀다.
오류 문구를 바꾸는 것만으로는 재시도 정책이 되지 않는다.

백오프도 횟수를 다 쓰면 실패가 모델로 넘어간다. 그 경우는 모델에 맡긴 것과 같은 길을 간다.

### 빠져나갈 길이 있으면 모델은 포기한다

모델에 맡기는 정책은 과제 문구에도 좌우된다. "세지 못하면 모름이라고 쓴다"를 덧붙이고 같은 시드로 돌린다.

In [18]:
ESCAPE = TASK.replace("한 줄씩 답한다.", "한 줄씩 답한다. 세지 못한 낱말은 '낱말: 모름' 으로 쓴다.")
for task, label in [(TASK, "그대로"), (ESCAPE, "모름 허용")]:
    r = attempt("모델에 맡김", seed=0, task=task)
    print(f"{label:6}  {r['정답']}/8  턴 {r['턴']}  도구 실행 {r['도구 실행']}  |"
          f" {(r['답'] or '').replace(chr(10), ' / ')[:90]}")

그대로     8/8  턴 4  도구 실행 14  | 한국독립당: 16 / 광복군: 13 / 독립신문: 11 / 하와이: 11 / 의열단: 10 / 신민회: 9 / 조선민족혁명당: 8 / 신한청년당: 6
모름 허용   3/8  턴 2  도구 실행 8  | 한국독립당: 16 / 광복군: 13 / 독립신문: 11 / 하와이: 모름 / 의열단: 모름 / 신민회: 모름 / 조선민족혁명당: 모름 / 신한청년당: 모름


오류 문구가 다시 부르라고 해도, 모름이 허용되면 모델은 다시 부르지 않고 모름이라고 쓰고 끝낸다.
틀린 숫자를 지어내지 않은 것은 좋지만 성공률은 떨어진다. 재시도를 모델에 맡기면 같은 하네스라도 과제를 어떻게 쓰느냐에 따라 거동이 바뀐다.

### 부작용이 있는 도구를 재시도하면

재시도가 안전하려면 같은 호출을 두 번 해도 결과가 한 번 한 것과 같아야 한다. 읽기 도구는 그렇다. 쓰기 도구는 아니다.
가장 곤란한 실패는 일은 해 놓고 응답만 잃어버린 타임아웃이다. 하네스는 실패로 보고 다시 부르고, 기록은 두 번 남는다.

`append_log`는 한 줄을 덧붙인다. `LostReply`로 감싸면 정해진 확률로 쓴 뒤에 타임아웃을 낸다.

In [19]:
LOG = Path("work/log.txt")


def append_log(entry):
    LOG.parent.mkdir(exist_ok=True)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(entry + "\n")
    return "기록함"


def lost_reply(fn, seed, p=0.3):
    rng = random.Random(seed)

    def tool(**args):
        result = fn(**args)
        if rng.random() < p:
            raise ToolTimeout("응답을 받지 못했다")
        return result
    return tool

백오프 정책으로 여덟 줄을 쓴다. 모델 없이 하네스 쪽에서 바로 부른다.

In [20]:
use("지수 백오프")
CONFIG["retry"]["base"] = 0.01
shutil.rmtree("work", ignore_errors=True)
tool = with_retry(lost_reply(append_log, seed=0))
for w in TERMS:
    tool(entry=f"{w}: {TRUTH[w]}")
lines = LOG.read_text(encoding="utf-8").splitlines()
print(f"쓰려던 줄 {len(TERMS)}  남은 줄 {len(lines)}  중복 {len(lines) - len(set(lines))}")

쓰려던 줄 8  남은 줄 9  중복 1


### 멱등성(Idempotency) 키

같은 요청에는 같은 키를 붙이고, 받는 쪽은 이미 처리한 키면 일을 다시 하지 않고 지난 결과를 돌려준다.
키는 도구 이름과 인자로 만든다. 하네스가 다시 부를 때도, 모델이 같은 인자로 다시 부를 때도 같은 키가 된다.

키를 기억하는 쪽은 도구 뒤의 저장소다. 하네스가 기억하면 하네스가 다시 시작될 때 잊는다. 여기서는 파일 하나로 흉내 낸다.

In [21]:
SEEN = Path("work/seen.json")


def idempotent(name, fn):
    def tool(**args):
        key = hashlib.sha256(json.dumps([name, args], ensure_ascii=False, sort_keys=True).encode()).hexdigest()[:16]
        seen = json.loads(SEEN.read_text()) if SEEN.exists() else {}
        if key in seen:
            return seen[key]
        result = fn(**args)
        seen[key] = result
        SEEN.write_text(json.dumps(seen))
        return result
    return tool

키 확인은 쓰기 바로 앞, 결과 저장은 쓰기 바로 뒤에 둔다. 응답을 잃는 지점은 그보다 바깥이다. 실제 서비스라면 이 부분이 받는 쪽 서버 안에 있다.

In [22]:
shutil.rmtree("work", ignore_errors=True)
tool = with_retry(lost_reply(idempotent("append_log", append_log), seed=0))
for w in TERMS:
    tool(entry=f"{w}: {TRUTH[w]}")
lines = LOG.read_text(encoding="utf-8").splitlines()
print(f"쓰려던 줄 {len(TERMS)}  남은 줄 {len(lines)}  중복 {len(lines) - len(set(lines))}")

쓰려던 줄 8  남은 줄 8  중복 0


대가도 있다. 인자로 키를 만들면, 같은 내용을 일부러 두 번 쓰려는 요청도 한 번으로 합쳐진다.
그런 요청이 정상인 도구라면 키에 요청 번호를 넣어야 하고, 그 번호는 재시도 사이에 바뀌지 않아야 한다.

In [23]:
shutil.rmtree("work", ignore_errors=True)

### 어떤 값을 넣는가

다시 불러 성공할 수 있는 실패와 그렇지 않은 실패를 먼저 가른다. 인자 오류는 재시도하지 않고 모델에게 돌려준다.
다시 불러 성공할 수 있는 실패는 하네스가 백오프로 처리한다. 특히 429 처럼 기다려야 하는 실패를 모델에 넘기면 턴과 토큰을 쓰면서도 넘기지 못한다.
대기 시간은 시간 예산(`max_seconds`)에서 나간다. 재시도 횟수와 대기 시간을 늘릴수록 한 번 실행이 길어지므로 두 값은 함께 정한다.

부작용이 있는 도구는 재시도 정책을 켜기 전에 멱등성부터 확인한다. 응답만 잃은 실패를 하네스는 구별할 수 없다.

### 정리

이 노트북에서 정한 것은 재시도를 어떻게 짜느냐가 아니라 다음 세 가지다. `with_retry`는 코어를 고치지 않으려고 고른 수단일 뿐이다.

1. 누가 재시도하는가. **기다려야 풀리는 실패는 하네스가 맡는다.** 모델은 기다릴 수 없어서, 다시 부르라는 오류 문구를 받아도 쿨다운 안에서 같은 실패를 되풀이하고, 빠져나갈 길이 있으면 포기한다.
2. 무엇을 재시도하는가. **타임아웃, 503, 429 처럼 같은 인자로 다시 불러 성공할 수 있는 실패만 재시도**한다. 인자 오류는 모델에게 돌려준다.
3. 재시도해도 되는가. 쓰기 도구는 일을 해 놓고 응답만 잃은 실패를 다시 실행해 결과가 두 번 남는다. 재시도를 켜기 전에 **멱등성 키**를 먼저 갖춘다.

### 비용 확인

In [24]:
print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${dollars(USAGE):.4f}")

호출 106회
입력 55,104 토큰   출력 6,283 토큰
누적 $0.2320
